## 1. 펑션콜링의 이해

오픈AI의 펑션콜링: https://platform.openai.com/docs/guides/function-calling

<img src="https://static.wikidocs.net/images/page/362007/tool_use_nH0htXB.PNG" width=500>

In [ ]:
# --- 가상의 날씨 데이터 (키는 영문 소문자) ---
_FAKE_WEATHER_DB = {
    "paris":  {"temperature": 14, "condition": "흐림"},
    "seoul":  {"temperature": 9,  "condition": "맑음"},
    "tokyo":  {"temperature": 17, "condition": "비"},
    "london": {"temperature": 11, "condition": "안개"},
}

def get_weather(location):
    key = location.strip().lower()
    data = _FAKE_WEATHER_DB.get(key, {"temperature": 20, "condition": "맑음"})
    return json.dumps(data, ensure_ascii=False)

In [ ]:
# 키는 Colab 왼쪽 🔑(보안 비밀)에 OPENAI_API_KEY 이름으로 저장하세요
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [ ]:
from openai import OpenAI
import json

client = OpenAI()

In [ ]:
# 사용할 함수 명세
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "특정 도시의 현재 날씨를 조회합니다. location에는 영문 도시명을 넣습니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "영문 도시명 (예: Paris, Seoul, Tokyo)",
                    },
                },
                "required": ["location"],
            },
        },
    },
]

# 사용자의 입력
messages = [
    {"role": "user", "content": "파리의 날씨는 어때?"},
]

In [ ]:
# 도구를 정의한 상태로 모델 호출
response = client.chat.completions.create(
    model="gpt-4.1",
    messages=messages, # 사용자의 입력
    tools=tools, # 함수 명세
    tool_choice="auto",
)

# GPT-4.1이 함수 명세와 사용자 입력을 바탕으로 선택한 호출할 함수와 파라미터.
assistant_message = response.choices[0].message

In [ ]:
print(assistant_message.tool_calls)

[ChatCompletionMessageFunctionToolCall(id='call_LCifsQ5wUKCLxsoBoBeY7ZuG', function=Function(arguments='{"location":"Paris"}', name='get_weather'), type='function')]


In [ ]:
print('GPT-4.1이 선택한 함수명')
print(assistant_message.tool_calls[0].function.name)

GPT-4.1이 선택한 함수명
get_weather


In [ ]:
print('GPT-4.1이 선택한 해당 함수의 파라미터')
print(assistant_message.tool_calls[0].function.arguments)

GPT-4.1이 선택한 해당 함수의 파라미터
{"location":"Paris"}


In [ ]:
# GPT-4.1의 응답을 이전 대화에 포함
messages.append(assistant_message)
print(messages)

[{'role': 'user', 'content': '파리의 날씨는 어때?'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_LCifsQ5wUKCLxsoBoBeY7ZuG', function=Function(arguments='{"location":"Paris"}', name='get_weather'), type='function')])]


In [ ]:
# 함수 실행 후 결과를 role="tool" 메시지로 전달
if assistant_message.tool_calls:
    for tool_call in assistant_message.tool_calls:

        # GPT-4.1이 선택한 함수와 파라미터를 바탕으로 실제 함수를 실행하여 함수 호출 결과를 얻음.
        if tool_call.function.name == "get_weather":
            location = json.loads(tool_call.function.arguments)["location"]

            # 실제 파싱한 함수명과 함수 파라미터를 바탕으로 실제 실행
            weather = get_weather(location)

            # 함수 호출 결과도 이전 대화에 포함
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": weather,
            })

In [ ]:
# messages는 지금까지 일어난 모든 대화 내역 내지는 대화 흐름을 기록.
# 사용자의 질문
# GPT-4.1이 제안한 함수명과 파라미터
# 실제 함수 호출 결과
print(messages)

[{'role': 'user', 'content': '파리의 날씨는 어때?'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_LCifsQ5wUKCLxsoBoBeY7ZuG', function=Function(arguments='{"location":"Paris"}', name='get_weather'), type='function')]), {'role': 'tool', 'tool_call_id': 'call_LCifsQ5wUKCLxsoBoBeY7ZuG', 'content': '{"temperature": 14, "condition": "흐림"}'}]


In [ ]:
# 사용자 입력, 함수와 파라미터 선택, 함수 호출 결과를 바탕으로 최종 응답 생성
second_response = client.chat.completions.create(
    model="gpt-4.1",
    messages=messages,
)

print(second_response.choices[0].message.content)

현재 파리의 날씨는 흐리고, 기온은 약 14도입니다. 옷차림에 참고하시기 바랍니다!


## 2. 데이터 생성 및 함수 생성

In [ ]:
pip install openai gradio

In [ ]:
import os
import json
import pandas as pd
from datetime import datetime
from typing import Dict, List, Optional, Union
from openai import OpenAI
import gradio as gr

# 샘플 데이터 (실제 구현에서는 DB나 API에서 가져올 것입니다)
products_data = [
    {"상품ID": "P001", "상품명": "프리미엄 강아지 간식 세트", "물류창고ID": "W02", "카테고리": "반려용품", "재고수량": 36, "가격": 32417, "입고일자": "2022-01-14"},
    {"상품ID": "P002", "상품명": "고양이 캣타워 XL", "물류창고ID": "W02", "카테고리": "반려용품", "재고수량": 91, "가격": 47747, "입고일자": "2023-03-07"},
    {"상품ID": "P003", "상품명": "애완동물 자동 급식기", "물류창고ID": "W02", "카테고리": "반려용품", "재고수량": 10, "가격": 22352, "입고일자": "2022-03-04"},
    {"상품ID": "P004", "상품명": "반려동물 이동가방", "물류창고ID": "W03", "카테고리": "반려용품", "재고수량": 99, "가격": 91542, "입고일자": "2021-08-01"},
    {"상품ID": "P005", "상품명": "펫 전용 샴푸", "물류창고ID": "W03", "카테고리": "반려용품", "재고수량": 49, "가격": 41276, "입고일자": "2020-12-26"}
]

customers_data = [
    {"고객ID": "C001", "고객명": "박예준", "생년월일": "2001-04-09", "성별": "M", "전화번호": "017-612-4538", "이메일": "sbag@example.com", "가입일자": "2020-08-10", "등급": "VIP", "포인트": 5010},
    {"고객ID": "C002", "고객명": "김미숙", "생년월일": "1975-10-05", "성별": "M", "전화번호": "052-368-8400", "이메일": "ohwang@example.org", "가입일자": "2024-01-23", "등급": "VIP", "포인트": 5771},
    {"고객ID": "C003", "고객명": "김예준", "생년월일": "2001-02-20", "성별": "M", "전화번호": "061-192-5591", "이메일": "gimyejun@example.org", "가입일자": "2020-07-10", "등급": "일반", "포인트": 4119},
    {"고객ID": "C004", "고객명": "곽하은", "생년월일": "1985-12-19", "성별": "M", "전화번호": "053-366-1601", "이메일": "iyeji@example.org", "가입일자": "2021-06-17", "등급": "VIP", "포인트": 4641},
    {"고객ID": "C005", "고객명": "김도윤", "생년월일": "1991-02-01", "성별": "M", "전화번호": "017-173-4341", "이메일": "gimsugja@example.net", "가입일자": "2023-09-17", "등급": "일반", "포인트": 4434}
]

orders_data = [
    {"주문ID": "O0001", "고객ID": "C084", "상품ID": "P011", "수량": 2, "주문일자": "2025-02-25", "결제수단": "카드", "결제상태": "보류", "배송ID": "D0001", "총주문액": 144210, "사용포인트": 640, "할인액": 10848, "배송비": 0, "최종결제액": 132722},
    {"주문ID": "O0002", "고객ID": "C025", "상품ID": "P040", "수량": 1, "주문일자": "2025-01-08", "결제수단": "휴대폰결제", "결제상태": "결제완료", "배송ID": "D0002", "총주문액": 87735, "사용포인트": 130, "할인액": 13728, "배송비": 0, "최종결제액": 73877},
    {"주문ID": "O0003", "고객ID": "C064", "상품ID": "P003", "수량": 1, "주문일자": "2025-01-26", "결제수단": "휴대폰결제", "결제상태": "결제완료", "배송ID": "D0003", "총주문액": 22352, "사용포인트": 510, "할인액": 1030, "배송비": 3000, "최종결제액": 23812},
    {"주문ID": "O0004", "고객ID": "C015", "상품ID": "P020", "수량": 1, "주문일자": "2025-02-04", "결제수단": "휴대폰결제", "결제상태": "취소", "배송ID": "D0004", "총주문액": 77873, "사용포인트": 841, "할인액": 14649, "배송비": 0, "최종결제액": 62383},
    {"주문ID": "O0005", "고객ID": "C032", "상품ID": "P007", "수량": 3, "주문일자": "2025-02-11", "결제수단": "계좌이체", "결제상태": "취소", "배송ID": "D0005", "총주문액": 208998, "사용포인트": 366, "할인액": 2840, "배송비": 0, "최종결제액": 205792},
    {"주문ID": "O0006", "고객ID": "C001", "상품ID": "P001", "수량": 1, "주문일자": "2025-03-01", "결제수단": "카드", "결제상태": "결제완료", "배송ID": "D0006", "총주문액": 32417, "사용포인트": 500, "할인액": 3242, "배송비": 0, "최종결제액": 28675},
    {"주문ID": "O0007", "고객ID": "C001", "상품ID": "P002", "수량": 1, "주문일자": "2025-03-10", "결제수단": "카드", "결제상태": "결제완료", "배송ID": "D0007", "총주문액": 47747, "사용포인트": 1000, "할인액": 4775, "배송비": 0, "최종결제액": 41972},
    {"주문ID": "O0008", "고객ID": "C001", "상품ID": "P005", "수량": 2, "주문일자": "2025-03-15", "결제수단": "계좌이체", "결제상태": "결제완료", "배송ID": "D0008", "총주문액": 82552, "사용포인트": 0, "할인액": 8255, "배송비": 0, "최종결제액": 74297}
]

deliveries_data = [
    {"배송ID": "D0001", "배송상태": "배송중", "출고일자": "2025-03-17", "도착예정일": "2025-03-27", "배송사": "우체국택배", "송장번호": "3885733726"},
    {"배송ID": "D0002", "배송상태": "배송중", "출고일자": "2025-03-07", "도착예정일": "2025-03-25", "배송사": "CJ대한통운", "송장번호": "3845805571"},
    {"배송ID": "D0003", "배송상태": "배송중", "출고일자": "2025-03-20", "도착예정일": "2025-03-27", "배송사": "우체국택배", "송장번호": "9508993384"},
    {"배송ID": "D0004", "배송상태": "배송완료", "출고일자": "2025-01-24", "도착예정일": "2025-03-25", "배송사": "우체국택배", "송장번호": "2156991983"},
    {"배송ID": "D0005", "배송상태": "배송중", "출고일자": "2025-01-23", "도착예정일": "2025-03-28", "배송사": "한진택배", "송장번호": "8878855674"},
    {"배송ID": "D0006", "배송상태": "배송완료", "출고일자": "2025-03-02", "도착예정일": "2025-03-04", "배송사": "CJ대한통운", "송장번호": "1234567890"},
    {"배송ID": "D0007", "배송상태": "배송중", "출고일자": "2025-03-11", "도착예정일": "2025-03-14", "배송사": "한진택배", "송장번호": "2345678901"},
    {"배송ID": "D0008", "배송상태": "배송준비중", "출고일자": None, "도착예정일": "2025-03-20", "배송사": "우체국택배", "송장번호": "3456789012"}
]

# 데이터프레임 생성
products_df = pd.DataFrame(products_data)
customers_df = pd.DataFrame(customers_data)
orders_df = pd.DataFrame(orders_data)
deliveries_df = pd.DataFrame(deliveries_data)

In [ ]:
# 1. 고객 프로필 정보 조회 함수
def get_customer_profile(customer_id: str) -> Dict:
    """
    고객 ID로 고객 프로필 정보를 조회합니다.

    Args:
        customer_id: 고객 ID

    Returns:
        고객 프로필 정보를 담은 사전 (없으면 빈 사전 반환)
    """
    customer = customers_df[customers_df['고객ID'] == customer_id]
    if customer.empty:
        return {}

    return customer.iloc[0].to_dict()

In [ ]:
# 2. 고객 주문 내역 조회 함수
def get_customer_orders(customer_id: str, start_date: Optional[str] = None, end_date: Optional[str] = None) -> List[Dict]:
    """
    고객 ID로 주문 내역을 조회합니다. 선택적으로 날짜 범위를 지정할 수 있습니다.

    Args:
        customer_id: 고객 ID
        start_date: 시작 날짜 (YYYY-MM-DD 형식, 선택사항)
        end_date: 종료 날짜 (YYYY-MM-DD 형식, 선택사항)

    Returns:
        주문 내역 목록 (없으면 빈 리스트 반환)
    """
    orders = orders_df[orders_df['고객ID'] == customer_id].copy()
    if orders.empty:
        return []

    # 날짜 필터링 (지정된 경우)
    if start_date:
        orders = orders[orders['주문일자'] >= start_date]
    if end_date:
        orders = orders[orders['주문일자'] <= end_date]

    # 상품 정보 추가
    result = []
    for _, order in orders.iterrows():
        order_dict = order.to_dict()

        # 상품 정보 가져오기
        product = products_df[products_df['상품ID'] == order['상품ID']]
        if not product.empty:
            order_dict['상품명'] = product.iloc[0]['상품명']
            order_dict['가격'] = product.iloc[0]['가격']

        result.append(order_dict)

    return result

In [ ]:
# 3. 고객 배송 상태 조회 함수
def get_customer_deliveries(customer_id: str) -> List[Dict]:
    """
    고객 ID로 배송 상태를 조회합니다.

    Args:
        customer_id: 고객 ID

    Returns:
        배송 정보 목록 (없으면 빈 리스트 반환)
    """
    # 고객의 주문 ID 가져오기
    orders = orders_df[orders_df['고객ID'] == customer_id]
    if orders.empty:
        return []

    # 주문에 해당하는 배송 정보 가져오기
    result = []
    for _, order in orders.iterrows():
        delivery_id = order['배송ID']
        delivery = deliveries_df[deliveries_df['배송ID'] == delivery_id]

        if not delivery.empty:
            delivery_dict = delivery.iloc[0].to_dict()
            delivery_dict['주문ID'] = order['주문ID']
            delivery_dict['상품ID'] = order['상품ID']

            # 상품 정보 추가
            product = products_df[products_df['상품ID'] == order['상품ID']]
            if not product.empty:
                delivery_dict['상품명'] = product.iloc[0]['상품명']

            result.append(delivery_dict)

    return result

In [ ]:
# 4. 고객 포인트 정보 조회 함수
def get_customer_points(customer_id: str) -> Dict:
    """
    고객 ID로 포인트 정보를 조회합니다.

    Args:
        customer_id: 고객 ID

    Returns:
        포인트 정보를 담은 사전 (없으면 빈 사전 반환)
    """
    customer = customers_df[customers_df['고객ID'] == customer_id]
    if customer.empty:
        return {}

    return {
        "고객ID": customer_id,
        "고객명": customer.iloc[0]['고객명'],
        "등급": customer.iloc[0]['등급'],
        "현재포인트": customer.iloc[0]['포인트'],
        "가입일자": customer.iloc[0]['가입일자']
    }

In [ ]:
# 5. 결제 정보 조회 함수
def get_payment_info(customer_id: str, order_id: Optional[str] = None) -> Union[Dict, List[Dict]]:
    """
    고객 ID와 선택적으로 주문 ID로 결제 정보를 조회합니다.

    Args:
        customer_id: 고객 ID
        order_id: 주문 ID (선택사항)

    Returns:
        결제 정보를 담은 사전 또는 사전 목록 (없으면 빈 사전/리스트 반환)
    """
    orders = orders_df[orders_df['고객ID'] == customer_id].copy()
    if orders.empty:
        return {} if order_id else []

    # 특정 주문 ID가 지정된 경우
    if order_id:
        order = orders[orders['주문ID'] == order_id]
        if order.empty:
            return {}

        return {
            "주문ID": order_id,
            "결제수단": order.iloc[0]['결제수단'],
            "결제상태": order.iloc[0]['결제상태'],
            "총주문액": order.iloc[0]['총주문액'],
            "사용포인트": order.iloc[0]['사용포인트'],
            "할인액": order.iloc[0]['할인액'],
            "배송비": order.iloc[0]['배송비'],
            "최종결제액": order.iloc[0]['최종결제액'],
            "주문일자": order.iloc[0]['주문일자']
        }

    # 모든 주문의 결제 정보 반환
    result = []
    for _, order in orders.iterrows():
        result.append({
            "주문ID": order['주문ID'],
            "결제수단": order['결제수단'],
            "결제상태": order['결제상태'],
            "총주문액": order['총주문액'],
            "사용포인트": order['사용포인트'],
            "할인액": order['할인액'],
            "배송비": order['배송비'],
            "최종결제액": order['최종결제액'],
            "주문일자": order['주문일자']
        })

    return result

In [ ]:
# 테스트용 고객 ID (실제 구현에서는 로그인한 사용자의 ID를 사용)
customer_id = "C001"

# 프로필 정보 조회
profile = get_customer_profile(customer_id)
print(f"고객 프로필: {profile}")

# 주문 내역 조회
orders = get_customer_orders(customer_id)
print(f"주문 내역: {orders}")

# 배송 상태 조회
deliveries = get_customer_deliveries(customer_id)
print(f"배송 상태: {deliveries}")

# 포인트 정보 조회
points = get_customer_points(customer_id)
print(f"포인트 정보: {points}")

# 결제 정보 조회
payment_info = get_payment_info(customer_id)
print(f"결제 정보: {payment_info}")

고객 프로필: {'고객ID': 'C001', '고객명': '박예준', '생년월일': '2001-04-09', '성별': 'M', '전화번호': '017-612-4538', '이메일': 'sbag@example.com', '가입일자': '2020-08-10', '등급': 'VIP', '포인트': 5010}
주문 내역: [{'주문ID': 'O0006', '고객ID': 'C001', '상품ID': 'P001', '수량': 1, '주문일자': '2025-03-01', '결제수단': '카드', '결제상태': '결제완료', '배송ID': 'D0006', '총주문액': 32417, '사용포인트': 500, '할인액': 3242, '배송비': 0, '최종결제액': 28675, '상품명': '프리미엄 강아지 간식 세트', '가격': np.int64(32417)}, {'주문ID': 'O0007', '고객ID': 'C001', '상품ID': 'P002', '수량': 1, '주문일자': '2025-03-10', '결제수단': '카드', '결제상태': '결제완료', '배송ID': 'D0007', '총주문액': 47747, '사용포인트': 1000, '할인액': 4775, '배송비': 0, '최종결제액': 41972, '상품명': '고양이 캣타워 XL', '가격': np.int64(47747)}, {'주문ID': 'O0008', '고객ID': 'C001', '상품ID': 'P005', '수량': 2, '주문일자': '2025-03-15', '결제수단': '계좌이체', '결제상태': '결제완료', '배송ID': 'D0008', '총주문액': 82552, '사용포인트': 0, '할인액': 8255, '배송비': 0, '최종결제액': 74297, '상품명': '펫 전용 샴푸', '가격': np.int64(41276)}]
배송 상태: [{'배송ID': 'D0006', '배송상태': '배송완료', '출고일자': '2025-03-02', '도착예정일': '2025-03-04', '배송사': 'C

## 3. 함수 명세 작성

In [ ]:
# OpenAI 클라이언트 설정
client = OpenAI()

In [ ]:
# 함수 매핑 - 실제 함수를 호출하는 역할
def execute_function(function_name, arguments):
    function_map = {
        "get_customer_profile": get_customer_profile,
        "get_customer_orders": get_customer_orders,
        "get_customer_deliveries": get_customer_deliveries,
        "get_customer_points": get_customer_points,
        "get_payment_info": get_payment_info
    }

    if function_name not in function_map:
        return {"error": f"Function {function_name} not found"}

    try:
        # customer_id가 비어있는지 확인
        if "customer_id" in arguments and (arguments["customer_id"] is None or arguments["customer_id"] == ""):
            arguments["customer_id"] = "C001"  # 기본값 설정

        # 함수 실행
        result = function_map[function_name](**arguments)

        # int64, float64 등의 numpy 타입을 일반 Python 타입으로 변환
        if isinstance(result, dict):
            for key, value in result.items():
                if hasattr(value, 'item'):  # numpy 숫자 타입 확인
                    result[key] = value.item()  # Python 내장 타입으로 변환
        elif isinstance(result, list):
            for item in result:
                if isinstance(item, dict):
                    for key, value in item.items():
                        if hasattr(value, 'item'):
                            item[key] = value.item()

        return result
    except Exception as e:
        return {"error": str(e)}

In [ ]:
# OpenAI 함수 호출 정의
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_customer_profile",
            "description": "고객 ID로 고객 프로필 정보를 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "customer_id": {
                        "type": "string",
                        "description": "고객 ID (예: C001)"
                    }
                },
                "required": ["customer_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_customer_orders",
            "description": "고객 ID로 주문 내역을 조회합니다. 선택적으로 날짜 범위를 지정할 수 있습니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "customer_id": {
                        "type": "string",
                        "description": "고객 ID (예: C001)"
                    },
                    "start_date": {
                        "type": "string",
                        "description": "시작 날짜 (YYYY-MM-DD 형식, 선택사항)"
                    },
                    "end_date": {
                        "type": "string",
                        "description": "종료 날짜 (YYYY-MM-DD 형식, 선택사항)"
                    }
                },
                "required": ["customer_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_customer_deliveries",
            "description": "고객 ID로 배송 상태를 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "customer_id": {
                        "type": "string",
                        "description": "고객 ID (예: C001)"
                    }
                },
                "required": ["customer_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_customer_points",
            "description": "고객 ID로 포인트 정보를 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "customer_id": {
                        "type": "string",
                        "description": "고객 ID (예: C001)"
                    }
                },
                "required": ["customer_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_payment_info",
            "description": "고객 ID와 선택적으로 주문 ID로 결제 정보를 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "customer_id": {
                        "type": "string",
                        "description": "고객 ID (예: C001)"
                    },
                    "order_id": {
                        "type": "string",
                        "description": "주문 ID (예: O0001, 선택사항)"
                    }
                },
                "required": ["customer_id"]
            }
        }
    }
]

## 4. 펑션콜링 동작

In [ ]:
# 대화 기록을 저장할 변수
conversation_history = {}
execution_logs = {}

In [ ]:
def process_message(message, customer_id, conversation_id=None):
    global conversation_history, execution_logs

    # 새로운 대화인 경우 대화 ID 생성 및 초기화
    if conversation_id is None or conversation_id not in conversation_history:
        conversation_id = f"conv_{len(conversation_history) + 1}"
        conversation_history[conversation_id] = [
            {"role": "system", "content": f"현재 고객 ID는 {customer_id}입니다. 이 고객의 정보를 조회할 때는 항상 이 ID를 사용하세요."}
        ]
        execution_logs[conversation_id] = ""

    # 사용자 메시지 추가
    conversation_history[conversation_id].append({"role": "user", "content": message})

    # 로그 초기화
    current_log = f"=== 새 요청 ===\n사용자 메시지: {message}\n고객 ID: {customer_id}\n\n"

    # 메시지 목록 구성
    messages = conversation_history[conversation_id]

    try:
        # OpenAI API 호출
        current_log += ">>> OpenAI API 호출 중...\n"
        response = client.chat.completions.create(
            model="gpt-4.1",
            messages=messages, # 사용자의 입력
            tools=tools, # 함수 명세
            tool_choice="auto"
        )

        # 응답 추출
        assistant_message = response.choices[0].message
        current_log += f">>> API 응답 수신 완료\n\n"

        # 함수 호출이 필요한 경우
        if assistant_message.tool_calls:
            current_log += f">>> 도구 호출 감지: {len(assistant_message.tool_calls)}개\n\n"

            # 응답에 함수 호출 정보 저장
            conversation_history[conversation_id].append(assistant_message)

            # 모든 함수 호출 수행
            for i, tool_call in enumerate(assistant_message.tool_calls, 1):

                # 함수명과 파라미터를 파싱 -> 아래의 execute_function에서 실제 실행할 예정
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)

                current_log += f"--- 도구 호출 #{i} ---\n"
                current_log += f"함수명: {function_name}\n"
                current_log += f"인자: {json.dumps(function_args, ensure_ascii=False, indent=2)}\n"

                # customer_id가 명시되지 않은 경우 현재 고객 ID 사용
                if "customer_id" in function_args and not function_args["customer_id"]:
                    function_args["customer_id"] = customer_id

                # 위에서 파싱한 함수명과 파라미터로 실제 함수 실행. 함수 실행 결과 가져오기
                function_response = execute_function(function_name, function_args)

                # 함수 호출 결과를 log에다가 추가로 저장.
                current_log += f"결과: {json.dumps(function_response, ensure_ascii=False, indent=2)}\n\n"

                # 함수 결과를 대화 기록에 추가
                conversation_history[conversation_id].append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": json.dumps(function_response, ensure_ascii=False)
                })

            # 함수 결과를 바탕으로 최종 응답 생성
            current_log += ">>> 최종 응답 생성 중...\n"
            second_response = client.chat.completions.create(
                model="gpt-4.1",
                messages=conversation_history[conversation_id]
            )

            final_response = second_response.choices[0].message.content
            current_log += f">>> 최종 응답 생성 완료\n\n"

            # 최종 응답을 대화 기록에 추가
            conversation_history[conversation_id].append({
                "role": "assistant",
                "content": final_response
            })

            execution_logs[conversation_id] += current_log
            return final_response, conversation_id, execution_logs[conversation_id]
        else:
            # 일반 응답인 경우
            content = assistant_message.content
            current_log += ">>> 일반 응답 (도구 호출 없음)\n"
            current_log += f"응답: {content}\n\n"

            # 응답을 대화 기록에 추가
            conversation_history[conversation_id].append({
                "role": "assistant",
                "content": content
            })

            execution_logs[conversation_id] += current_log
            return content, conversation_id, execution_logs[conversation_id]

    except Exception as e:
        error_message = f"오류가 발생했습니다: {str(e)}"
        current_log += f">>> 오류 발생: {str(e)}\n"
        execution_logs[conversation_id] += current_log
        return error_message, conversation_id, execution_logs[conversation_id]

## 4. 그라디오

In [ ]:
import gradio as gr


def _new_chatbot(**kwargs):
    """설치된 gradio 버전에 맞게 Chatbot을 만들고, 실제 히스토리 포맷을 함께 반환."""
    try:
        cb = gr.Chatbot(type="messages", **kwargs)   # gradio 4.44 ~ 5.x
    except TypeError:
        cb = gr.Chatbot(**kwargs)                    # type 인자가 없는 버전 (3.x 또는 6.x+)

    # 인자 수용 여부가 아니라 컴포넌트의 실제 포맷을 읽는다
    fmt = getattr(cb, "type", None)
    if fmt not in ("messages", "tuples"):
        try:
            major = int(str(gr.__version__).split(".")[0])
        except Exception:
            major = 5
        fmt = "tuples" if major < 5 else "messages"  # 5 미만은 tuples, 이후는 messages
    return cb, fmt


def create_ui():
    with gr.Blocks(title="고객 서비스 챗봇") as demo:
        gr.HTML("<style>footer{visibility:hidden}</style>")
        gr.Markdown("# 고객 서비스 챗봇")
        gr.Markdown("고객 ID를 입력하고 문의사항을 입력하세요.")

        with gr.Row():
            with gr.Column(scale=2):
                customer_id_input = gr.Textbox(label="고객 ID", placeholder="예: C001", value="C001")
                chatbot, chat_format = _new_chatbot(label="대화", height=400)
                msg = gr.Textbox(label="메시지", placeholder="무엇을 도와드릴까요?")
                clear = gr.Button("대화 초기화")

            with gr.Column(scale=1):
                logs = gr.Textbox(label="실행 로그", lines=25, max_lines=25)

        conversation_id = gr.State(value=None)

        def user_input(message, history, customer_id, conv_id):
            history = list(history or [])

            if not message or not message.strip():
                return gr.update(value=""), history, conv_id, gr.update()

            if not customer_id or not customer_id.strip():
                customer_id = "C001"

            response, new_conv_id, log = process_message(message, customer_id, conv_id)
            response = "" if response is None else str(response)

            if chat_format == "messages":
                history += [
                    {"role": "user", "content": message},
                    {"role": "assistant", "content": response},
                ]
            else:
                history.append((message, response))

            return gr.update(value=""), history, new_conv_id, log

        msg.submit(
            user_input,
            [msg, chatbot, customer_id_input, conversation_id],
            [msg, chatbot, conversation_id, logs],
        )

        clear.click(lambda: ([], None, ""), [], [chatbot, conversation_id, logs])

    print(f"[gradio {gr.__version__}] chat_format = {chat_format}")
    return demo

In [ ]:
demo = create_ui()
demo.launch(debug=True)

[gradio 6.31.0] chat_format = messages
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://af994ce581e371963b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


위 코드들을 모두 클로드에게 전달한 뒤  

' 이번에는 ui를 html css 사용하는 기깔나는 걸로 하고 싶다.

셀 안에서 html 코드 작성해서 파일로 저장하는 방식 등을 써서라도

자바스크립트 ui 버전 코드 그 아래 이어서 만들어.'  

라고 작성해서 얻은 코드가 아래와 같다.

In [ ]:
!pip install -q fastapi "uvicorn[standard]"

In [ ]:
from pathlib import Path

UI_DIR = Path("webui")
UI_DIR.mkdir(exist_ok=True)

INDEX_HTML = """<!DOCTYPE html>
<html lang="ko">
<head>
<meta charset="utf-8" />
<meta name="viewport" content="width=device-width, initial-scale=1" />
<title>CS Assistant</title>
<style>
  :root{
    --bg:#0b0f1a;
    --panel:rgba(255,255,255,.045);
    --line:rgba(255,255,255,.09);
    --txt:#e8ecf6;
    --dim:#8a93a8;
    --a1:#7c5cff;
    --a2:#22d3ee;
  }
  *{box-sizing:border-box;margin:0;padding:0}
  html,body{height:100%}
  body{
    font-family:"Noto Sans KR","Pretendard",-apple-system,"Segoe UI",sans-serif;
    color:var(--txt); background:var(--bg);
    background-image:
      radial-gradient(900px 500px at 8% -10%, rgba(124,92,255,.28), transparent 60%),
      radial-gradient(800px 500px at 100% 0%, rgba(34,211,238,.18), transparent 55%),
      radial-gradient(700px 600px at 50% 120%, rgba(236,72,153,.14), transparent 60%);
    padding:22px; overflow:hidden;
  }
  .app{max-width:1240px;height:calc(100vh - 44px);margin:0 auto;display:flex;flex-direction:column;gap:16px}

  .topbar{display:flex;align-items:center;gap:14px;flex-wrap:wrap}
  .brand{display:flex;align-items:center;gap:11px}
  .logo{width:38px;height:38px;border-radius:12px;background:linear-gradient(135deg,var(--a1),var(--a2));
        display:grid;place-items:center;font-size:19px;box-shadow:0 6px 22px rgba(124,92,255,.45)}
  .brand h1{font-size:19px;font-weight:800;letter-spacing:-.4px}
  .brand p{font-size:11.5px;color:var(--dim);margin-top:1px}
  .spacer{flex:1}
  .field{display:flex;align-items:center;gap:8px;background:var(--panel);border:1px solid var(--line);
         border-radius:12px;padding:8px 13px;backdrop-filter:blur(10px)}
  .field label{font-size:11px;color:var(--dim);white-space:nowrap}
  .field input{background:none;border:none;outline:none;color:var(--txt);font-size:13px;
               font-weight:700;width:74px;letter-spacing:.5px}
  .btn{background:var(--panel);border:1px solid var(--line);color:var(--dim);border-radius:12px;
       padding:10px 15px;font-size:12.5px;cursor:pointer;transition:.18s;backdrop-filter:blur(10px)}
  .btn:hover{color:var(--txt);border-color:rgba(255,255,255,.24);transform:translateY(-1px)}

  .main{flex:1;display:grid;grid-template-columns:1.45fr 1fr;gap:16px;min-height:0}
  .panel{background:var(--panel);border:1px solid var(--line);border-radius:20px;
         backdrop-filter:blur(14px);display:flex;flex-direction:column;min-height:0;overflow:hidden}
  .phead{padding:13px 18px;border-bottom:1px solid var(--line);display:flex;align-items:center;gap:9px}
  .phead h2{font-size:12.5px;font-weight:700;letter-spacing:.3px}
  .dot{width:7px;height:7px;border-radius:50%;background:#34d399;box-shadow:0 0 10px #34d399;
       animation:pulse 2s infinite}
  @keyframes pulse{0%,100%{opacity:1}50%{opacity:.35}}

  .msgs{flex:1;overflow-y:auto;padding:20px;display:flex;flex-direction:column;gap:15px}
  .msg{display:flex;gap:10px;align-items:flex-end;animation:rise .32s cubic-bezier(.2,.8,.3,1)}
  @keyframes rise{from{opacity:0;transform:translateY(12px)}to{opacity:1;transform:none}}
  .msg.user{flex-direction:row-reverse}
  .av{width:32px;height:32px;border-radius:50%;flex-shrink:0;display:grid;place-items:center;font-size:14px}
  .bot .av{background:linear-gradient(135deg,var(--a1),var(--a2))}
  .user .av{background:rgba(255,255,255,.1);border:1px solid var(--line)}
  .bubble{max-width:76%;padding:12px 16px;font-size:13.5px;line-height:1.72;
          white-space:pre-wrap;word-break:break-word}
  .bot .bubble{background:rgba(255,255,255,.06);border:1px solid var(--line);
               border-radius:16px 16px 16px 5px}
  .user .bubble{background:linear-gradient(135deg,#6d5cff,#9333ea);color:#fff;
                border-radius:16px 16px 5px 16px;box-shadow:0 6px 20px rgba(109,92,255,.32)}
  .typing{display:flex;gap:5px;padding:15px 17px}
  .typing i{width:7px;height:7px;border-radius:50%;background:var(--dim);animation:bounce 1.3s infinite}
  .typing i:nth-child(2){animation-delay:.18s}
  .typing i:nth-child(3){animation-delay:.36s}
  @keyframes bounce{0%,60%,100%{transform:translateY(0);opacity:.4}30%{transform:translateY(-6px);opacity:1}}

  .chips{display:flex;gap:8px;padding:0 18px 12px;flex-wrap:wrap}
  .chip{background:rgba(255,255,255,.05);border:1px solid var(--line);color:var(--dim);
        border-radius:999px;padding:7px 13px;font-size:11.5px;cursor:pointer;transition:.18s}
  .chip:hover{color:var(--txt);border-color:rgba(124,92,255,.6);background:rgba(124,92,255,.14)}

  .composer{padding:13px 16px;border-top:1px solid var(--line);display:flex;gap:10px;align-items:flex-end}
  .composer textarea{flex:1;background:rgba(0,0,0,.24);border:1px solid var(--line);border-radius:14px;
    color:var(--txt);padding:12px 15px;font-size:13.5px;font-family:inherit;resize:none;outline:none;
    max-height:120px;line-height:1.6;transition:.18s}
  .composer textarea:focus{border-color:rgba(124,92,255,.65);box-shadow:0 0 0 3px rgba(124,92,255,.14)}
  .send{width:44px;height:44px;border-radius:14px;border:none;flex-shrink:0;cursor:pointer;
    background:linear-gradient(135deg,var(--a1),var(--a2));color:#fff;font-size:16px;transition:.18s;
    box-shadow:0 6px 18px rgba(124,92,255,.4)}
  .send:hover{transform:translateY(-2px) scale(1.04)}
  .send:disabled{opacity:.4;cursor:not-allowed;transform:none}

  .log{flex:1;overflow-y:auto;padding:15px 17px;font-family:"JetBrains Mono","D2Coding",Consolas,monospace;
       font-size:11.3px;line-height:1.78}
  .log .l{white-space:pre-wrap;word-break:break-all}
  .l-head{color:#f0abfc;font-weight:700;margin-top:8px}
  .l-call{color:var(--a2)}
  .l-sep{color:#475569}
  .l-fn{color:#c4b5fd;font-weight:700}
  .l-res{color:#6ee7b7}
  .l-err{color:#fb7185;font-weight:700}
  .l-def{color:#94a3b8}
  .empty{color:#4b5563;font-size:11.5px;padding:6px}

  ::-webkit-scrollbar{width:7px;height:7px}
  ::-webkit-scrollbar-thumb{background:rgba(255,255,255,.14);border-radius:99px}
  ::-webkit-scrollbar-thumb:hover{background:rgba(255,255,255,.26)}
  ::-webkit-scrollbar-track{background:transparent}

  @media(max-width:900px){.main{grid-template-columns:1fr;grid-template-rows:1.6fr 1fr}}
</style>
</head>
<body>
<div class="app">

  <div class="topbar">
    <div class="brand">
      <div class="logo">🐾</div>
      <div>
        <h1>고객 서비스 어시스턴트</h1>
        <p>Function Calling · GPT-4.1</p>
      </div>
    </div>
    <div class="spacer"></div>
    <div class="field">
      <label>고객 ID</label>
      <input id="cid" value="C001" maxlength="8" />
    </div>
    <button class="btn" id="reset">대화 초기화</button>
  </div>

  <div class="main">
    <section class="panel">
      <div class="phead"><span class="dot"></span><h2>CHAT</h2></div>
      <div class="msgs" id="msgs"></div>
      <div class="chips" id="chips"></div>
      <div class="composer">
        <textarea id="input" rows="1" placeholder="무엇을 도와드릴까요?  (Enter 전송 · Shift+Enter 줄바꿈)"></textarea>
        <button class="send" id="send">➤</button>
      </div>
    </section>

    <section class="panel">
      <div class="phead"><span class="dot" style="background:#fbbf24;box-shadow:0 0 10px #fbbf24"></span><h2>EXECUTION LOG</h2></div>
      <div class="log" id="log"><div class="empty">아직 실행된 함수 호출이 없습니다.</div></div>
    </section>
  </div>

</div>

<script>
const BASE = location.pathname.endsWith('/') ? location.pathname : location.pathname + '/';
const msgsEl = document.getElementById('msgs');
const logEl  = document.getElementById('log');
const input  = document.getElementById('input');
const sendBtn= document.getElementById('send');
const cidEl  = document.getElementById('cid');

let convId = null;
let busy = false;

const SUGGESTIONS = [
  '내 주문 내역 알려줘',
  '배송 어디까지 왔어?',
  '포인트 얼마나 남았어?',
  '결제 정보 확인해줘',
  '내 등급이 뭐야?'
];

function addMsg(role, text){
  const wrap = document.createElement('div');
  wrap.className = 'msg ' + (role === 'user' ? 'user' : 'bot');
  const av = document.createElement('div');
  av.className = 'av';
  av.textContent = role === 'user' ? '🙂' : '🐾';
  const b = document.createElement('div');
  b.className = 'bubble';
  b.textContent = text;
  wrap.appendChild(av); wrap.appendChild(b);
  msgsEl.appendChild(wrap);
  msgsEl.scrollTop = msgsEl.scrollHeight;
  return wrap;
}

function addTyping(){
  const wrap = document.createElement('div');
  wrap.className = 'msg bot';
  wrap.innerHTML = '<div class="av">🐾</div><div class="bubble typing"><i></i><i></i><i></i></div>';
  msgsEl.appendChild(wrap);
  msgsEl.scrollTop = msgsEl.scrollHeight;
  return wrap;
}

function classify(line){
  const t = line.trim();
  if (t.startsWith('===')) return 'l-head';
  if (t.startsWith('>>>')) return t.indexOf('오류') >= 0 ? 'l-err' : 'l-call';
  if (t.startsWith('---')) return 'l-sep';
  if (t.startsWith('함수명')) return 'l-fn';
  if (t.startsWith('인자') || t.startsWith('결과')) return 'l-res';
  if (t.indexOf('"error"') >= 0) return 'l-err';
  return 'l-def';
}

function renderLog(text){
  logEl.innerHTML = '';
  if (!text || !text.trim()){
    logEl.innerHTML = '<div class="empty">아직 실행된 함수 호출이 없습니다.</div>';
    return;
  }
  text.split('\\n').forEach(function(line){
    const d = document.createElement('div');
    d.className = 'l ' + classify(line);
    d.textContent = line;
    logEl.appendChild(d);
  });
  logEl.scrollTop = logEl.scrollHeight;
}

async function send(text){
  if (!text.trim() || busy) return;
  busy = true; sendBtn.disabled = true;

  addMsg('user', text);
  input.value = ''; input.style.height = 'auto';
  const typing = addTyping();

  try {
    const res = await fetch(BASE + 'api/chat', {
      method: 'POST',
      headers: {'Content-Type': 'application/json'},
      body: JSON.stringify({
        message: text,
        customer_id: (cidEl.value || 'C001').trim(),
        conversation_id: convId
      })
    });
    const data = await res.json();
    typing.remove();
    convId = data.conversation_id;
    addMsg('bot', data.response);
    renderLog(data.log);
  } catch (e) {
    typing.remove();
    addMsg('bot', '⚠️ 서버 통신 오류: ' + e.message);
  }

  busy = false; sendBtn.disabled = false;
  input.focus();
}

sendBtn.onclick = function(){ send(input.value); };

input.addEventListener('keydown', function(e){
  if (e.key === 'Enter' && !e.shiftKey){ e.preventDefault(); send(input.value); }
});
input.addEventListener('input', function(){
  input.style.height = 'auto';
  input.style.height = Math.min(input.scrollHeight, 120) + 'px';
});

document.getElementById('reset').onclick = async function(){
  try { await fetch(BASE + 'api/reset', {
    method: 'POST', headers: {'Content-Type':'application/json'},
    body: JSON.stringify({conversation_id: convId})
  }); } catch(e) {}
  convId = null;
  msgsEl.innerHTML = '';
  renderLog('');
  welcome();
};

const chipsEl = document.getElementById('chips');
SUGGESTIONS.forEach(function(s){
  const c = document.createElement('button');
  c.className = 'chip'; c.textContent = s;
  c.onclick = function(){ send(s); };
  chipsEl.appendChild(c);
});

function welcome(){
  addMsg('bot', '안녕하세요! 주문 · 배송 · 포인트 · 결제 정보를 조회해 드립니다.\\n아래 추천 질문을 누르거나 자유롭게 물어보세요.');
}
welcome();
input.focus();
</script>
</body>
</html>
"""

(UI_DIR / "index.html").write_text(INDEX_HTML, encoding="utf-8")
print(f"저장 완료 → {(UI_DIR / 'index.html').resolve()}  ({len(INDEX_HTML):,} bytes)")

저장 완료 → /content/webui/index.html  (11,132 bytes)


In [ ]:
import threading, time, uvicorn, urllib.request
from google.colab import output

from typing import Optional
from fastapi import FastAPI
from fastapi.responses import FileResponse
from pydantic import BaseModel


# FastAPI 객체 생성
api = FastAPI()


class ChatRequest(BaseModel):
    message: str
    customer_id: str = "C001"
    conversation_id: Optional[str] = None


class ResetRequest(BaseModel):
    conversation_id: Optional[str] = None


# 메인 HTML 화면 반환
@api.get("/")
def serve_index():
    return FileResponse(UI_DIR / "index.html")


# 채팅 요청 처리
@api.post("/api/chat")
def chat(request: ChatRequest):
    response, conversation_id, log = process_message(
        message=request.message,
        customer_id=request.customer_id,
        conversation_id=request.conversation_id,
    )

    return {
        "response": response,
        "conversation_id": conversation_id,
        "log": log,
    }


# 대화 초기화
@api.post("/api/reset")
def reset_chat(request: ResetRequest):
    if request.conversation_id:
        conversation_history.pop(request.conversation_id, None)
        execution_logs.pop(request.conversation_id, None)

    return {"success": True}


PORT = 8000
_server = None


def start_server(port=PORT):
    global _server

    if _server is not None:
        _server.should_exit = True
        time.sleep(1.2)

    config = uvicorn.Config(
        api,
        host="0.0.0.0",
        port=port,
        log_level="warning"
    )

    _server = uvicorn.Server(config)
    threading.Thread(target=_server.run, daemon=True).start()

    for _ in range(40):
        time.sleep(0.3)

        try:
            urllib.request.urlopen(
                f"http://127.0.0.1:{port}/",
                timeout=2
            )

            print(f"✅ 서버 기동 완료 (port {port})")
            return True

        except Exception:
            continue

    print(
        f"❌ 서버 기동 실패 (port {port}) "
        "— PORT를 8010 등으로 바꿔 재시도하세요."
    )
    return False


if start_server():
    print("\n" + "=" * 46)
    print("  아래 링크를 '클릭'하면 새 탭에서 열립니다")
    print("  (주소창에 직접 입력하면 안 됩니다)")
    print("=" * 46)

    output.serve_kernel_port_as_window(PORT)

    print("\n↓ 아래는 셀 안에 바로 띄운 화면입니다 ↓")
    output.serve_kernel_port_as_iframe(PORT, height=820)

✅ 서버 기동 완료 (port 8000)

  아래 링크를 '클릭'하면 새 탭에서 열립니다
  (주소창에 직접 입력하면 안 됩니다)
Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>


↓ 아래는 셀 안에 바로 띄운 화면입니다 ↓


<IPython.core.display.Javascript object>